# Step 2: Data Preprocessing and Validation Pipeline
**Brain MRI Classification Project**

This notebook verifies the data preprocessing pipeline, reproducible stratified train/validation split, input shape standardization, and conservative augmentation for brain tumor classification.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import torch

# Add project root to sys.path
project_root = Path("..").resolve()
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

from src.config import CLASSES, LABEL2ID, ID2LABEL, RANDOM_SEED, IMAGE_SIZE
from src.dataset import get_data_loaders
from src.preprocessing import denormalize_tensor

print(f"Target Image Dimensions: {IMAGE_SIZE}")
print(f"Label Mapping: {LABEL2ID}")
print(f"Fixed Seed: {RANDOM_SEED}")

In [ ]:
# Initialize DataLoaders
train_loader, val_loader, test_loader, stats = get_data_loaders(
    batch_size=16,
    num_workers=0,
    seed=RANDOM_SEED
)

print("=== Split Counts & Class Distribution ===")
for split, data in stats["splits"].items():
    print(f"\nSplit: {split.upper()} (Total: {data['total_samples']})")
    for cls, cnt in data["class_counts"].items():
        pct = data["class_distribution_percent"][cls]
        print(f"  - {cls} (ID {LABEL2ID[cls]}): {cnt} images ({pct:.2f}%)")

In [ ]:
# Extract one batch and check shapes
train_imgs, train_labels, train_files = next(iter(train_loader))
val_imgs, val_labels, val_files = next(iter(val_loader))

print("Train batch shape:", train_imgs.shape)
print("Validation batch shape:", val_imgs.shape)
print("Device availability - MPS:", torch.backends.mps.is_available())

In [ ]:
# Visualize Augmented Training vs Deterministic Validation Samples
train_vis = denormalize_tensor(train_imgs).permute(0, 2, 3, 1).cpu().numpy()
val_vis = denormalize_tensor(val_imgs).permute(0, 2, 3, 1).cpu().numpy()

fig, axes = plt.subplots(4, 4, figsize=(14, 14))
for class_id, class_name in enumerate(CLASSES):
    train_idx = (train_labels == class_id).nonzero(as_tuple=True)[0]
    val_idx = (val_labels == class_id).nonzero(as_tuple=True)[0]
    for col in range(2):
        ax = axes[class_id, col]
        if len(train_idx) > col:
            ax.imshow(train_vis[train_idx[col].item()])
            ax.set_title(f"[Train/Aug] {class_name}\nID: {class_id}", fontsize=8)
        ax.axis("off")
    for col in range(2):
        ax = axes[class_id, col + 2]
        if len(val_idx) > col:
            ax.imshow(val_vis[val_idx[col].item()])
            ax.set_title(f"[Val/Preproc] {class_name}\nID: {class_id}", fontsize=8)
        ax.axis("off")

plt.suptitle("Brain MRI Preprocessed Samples Verification", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()